# SybilGAT — Graph Attention Network for Twitter Bot Detection

Detects Twitter bots using a 2-layer GAT on cresci-2017. Proves that attention weights learn to suppress bot→human camouflage edges (p < 10⁻⁴¹).

**Dataset:** cresci-2017 — 14,368 users, 9 classes (1 genuine, 8 bot subtypes), 203K edges (k-NN graph, k=10)  
**Features:** 217 total — 17 profile features + 200 TF-IDF tweet features

## 1. Install Dependencies

In [ ]:
!pip install torch_geometric -q
# These four are optional accelerators — skip pyg_lib entirely
!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.10.0+cpu.html -q

## 2. Mount Drive & Load Data

Expects cresci-2017 at `/content/drive/MyDrive/cresci/`  
File structure: `group.csv/group.csv/users.csv` and `tweets.csv` (double-nested zip artifact)  
Tweet files require `encoding='latin-1'` due to emoji/non-Latin characters.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import kneighbors_graph
import torch
from torch_geometric.data import Data
from scipy.sparse import issparse

# ── CONFIG ────────────────────────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/cresci'
MAX_TWEETS = 200   # per user
KNN_K      = 10    # neighbors in graph
TFIDF_DIM  = 200

GROUPS = {
    'genuine_accounts':     0,  # human
    'fake_followers':       1,
    'social_spambots_1':    1,
    'social_spambots_2':    1,
    'social_spambots_3':    1,
    'traditional_spambots_1': 1,
    'traditional_spambots_2': 1,
    'traditional_spambots_3': 1,
    'traditional_spambots_4': 1,
}

# ── HELPERS ───────────────────────────────────────────────────────────────────
def path(group, fname):
    return os.path.join(BASE, f'{group}.csv', f'{group}.csv', fname)

def parse_age_days(created_at_series):
    ref = pd.Timestamp('2015-05-01', tz='UTC')
    dt  = pd.to_datetime(created_at_series, utc=True, errors='coerce')
    return (ref - dt).dt.total_seconds() / 86400

def build_profile_features(df):
    df = df.copy()
    df['age_days']    = parse_age_days(df['created_at']).clip(lower=1)
    df['has_url']     = df['url'].notna().astype(float)
    df['has_desc']    = df['description'].notna().astype(float)
    df['name_len']    = df['name'].fillna('').str.len().astype(float)
    df['screen_len']  = df['screen_name'].fillna('').str.len().astype(float)
    df['ff_ratio']    = df['followers_count'] / (df['friends_count'] + 1)
    df['tweet_rate']  = df['statuses_count']  / df['age_days']
    df['fav_rate']    = df['favourites_count'] / df['age_days']

    cols = [
        'followers_count', 'friends_count', 'statuses_count',
        'favourites_count', 'listed_count',
        'verified', 'geo_enabled', 'default_profile',
        'has_url', 'has_desc',
        'name_len', 'screen_len', 'age_days',
        'ff_ratio', 'tweet_rate', 'fav_rate',
        'statuses_count',   # n_tweets alias
    ]
    feat = df[cols].copy()
    feat = feat.apply(pd.to_numeric, errors='coerce').fillna(0)
    return feat.values.astype(np.float32)
def stream_tweet_text(group):
    """Stream tweets.csv in chunks, collect up to MAX_TWEETS per user."""
    fpath = path(group, 'tweets.csv')
    if not os.path.exists(fpath):
        return {}
    user_tweets = {}
    for chunk in pd.read_csv(fpath, chunksize=50_000,
                              usecols=['user_id', 'text'],
                              dtype={'user_id': str, 'text': str},
                              encoding='latin-1',        # ← fix
                              on_bad_lines='skip'):
        for uid, grp in chunk.groupby('user_id'):
            existing = user_tweets.get(uid, [])
            if len(existing) < MAX_TWEETS:
                texts = grp['text'].dropna().tolist()
                user_tweets[uid] = (existing + texts)[:MAX_TWEETS]
    return user_tweets

# ── LOAD ALL GROUPS ───────────────────────────────────────────────────────────
all_users  = []
all_labels = []

print("Loading groups...")
for group, label in GROUPS.items():
    upath = path(group, 'users.csv')
    if not os.path.exists(upath):
        print(f"  SKIP {group} (no users.csv)")
        continue

    df = pd.read_csv(upath, dtype={'id': str}, on_bad_lines='skip')
    df['_group'] = group
    df['_label'] = label
    all_users.append(df)
    print(f"  {group}: {len(df)} users, label={label}")

users_df = pd.concat(all_users, ignore_index=True)
print(f"\nTotal users: {len(users_df)}")
print(f"Bots: {(users_df['_label']==1).sum()}  Humans: {(users_df['_label']==0).sum()}")

# ── PROFILE FEATURES ──────────────────────────────────────────────────────────
print("\nBuilding profile features...")
profile_feat = build_profile_features(users_df)
print(f"Profile features shape: {profile_feat.shape}")

# ── TWEET FEATURES ────────────────────────────────────────────────────────────
print("\nStreaming tweets (this takes a few minutes for genuine_accounts)...")
all_tweet_text = {}
for group in GROUPS:
    if not os.path.exists(path(group, 'tweets.csv')):
        continue
    print(f"  Streaming {group}...")
    all_tweet_text.update(stream_tweet_text(group))
print(f"  Users with tweets: {len(all_tweet_text)}")

# Map user_id -> concatenated tweet string
uid_col = users_df['id'].astype(str).tolist()
corpus  = [' '.join(all_tweet_text.get(uid, [''])) for uid in uid_col]

print("Fitting TF-IDF...")
tfidf = TfidfVectorizer(max_features=TFIDF_DIM, sublinear_tf=True,
                         strip_accents='unicode', min_df=3)
tfidf_feat = tfidf.fit_transform(corpus)
tfidf_arr  = tfidf_feat.toarray().astype(np.float32)
print(f"TF-IDF shape: {tfidf_arr.shape}")

# ── COMBINE FEATURES ──────────────────────────────────────────────────────────
X = np.concatenate([profile_feat, tfidf_arr], axis=1)
y = users_df['_label'].values.astype(np.long)
print(f"\nFinal feature matrix: {X.shape}")

# ── BUILD k-NN GRAPH ──────────────────────────────────────────────────────────
print(f"\nBuilding {KNN_K}-NN graph...")
A = kneighbors_graph(X, n_neighbors=KNN_K, mode='connectivity',
                     include_self=False, n_jobs=-1)
A = A + A.T   # make symmetric (undirected)
A.data[:] = 1 # binarize

cx      = A.tocoo()
edge_index = torch.tensor(np.vstack([cx.row, cx.col]), dtype=torch.long)
print(f"Edges: {edge_index.shape[1]:,}")

# ── PyG DATA OBJECT ───────────────────────────────────────────────────────────
data = Data(
    x          = torch.tensor(X, dtype=torch.float),
    edge_index = edge_index,
    y          = torch.tensor(y, dtype=torch.long),
)

n = data.num_nodes
idx      = torch.randperm(n)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

data.train_mask = torch.zeros(n, dtype=torch.bool)
data.val_mask   = torch.zeros(n, dtype=torch.bool)
data.test_mask  = torch.zeros(n, dtype=torch.bool)
data.train_mask[idx[:train_end]]        = True
data.val_mask  [idx[train_end:val_end]] = True
data.test_mask [idx[val_end:]]          = True

print(f"\n✓ Graph ready: {data}")
print(f"  Train: {data.train_mask.sum()}  Val: {data.val_mask.sum()}  Test: {data.test_mask.sum()}")

## 3. Train SybilGAT + Baselines

Trains Logistic Regression, Random Forest, GCN, GraphSAGE, and SybilGAT.

| Model | Accuracy | Macro F1 |
|---|---|---|
| Logistic Regression | 93.46% | 91.31% |
| GCN | 96.89% | 95.81% |
| SybilGAT | 97.12% | 96.18% |
| GraphSAGE | 97.22% | 96.29% |
| Random Forest | 98.79% | 98.41% |

> RF's 98.4% reflects cresci-2017's synthetically distinct bot classes — documented in ACM WWW 2023 as a convenience sampling artifact, not a model failure.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# ── MODEL ─────────────────────────────────────────────────────────────────────
class SybilGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads,
                             dropout=dropout, concat=True)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1,
                             dropout=dropout, concat=False)
        self.skip = nn.Linear(in_channels, out_channels)
        self.bn1  = nn.BatchNorm1d(hidden_channels * heads)

    def forward(self, x, edge_index):
        x_in = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv1(x_in, edge_index)
        x    = self.bn1(x)
        x    = F.elu(x)
        x    = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv2(x, edge_index)
        x    = x + self.skip(x_in)
        return x

# ── DEVICE ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
data_d = data.to(device)

# ── CLASS WEIGHTS ─────────────────────────────────────────────────────────────
n_humans = (data.y == 0).sum().item()
n_bots   = (data.y == 1).sum().item()
# Upweight humans since they're 3x underrepresented
cw = torch.tensor([n_bots / n_humans, 1.0], dtype=torch.float).to(device)
print(f"Class weights: human={cw[0]:.2f}  bot={cw[1]:.2f}")

# ── TRAIN FUNCTION ────────────────────────────────────────────────────────────
def train_gat(data, device, epochs=300, hidden=64, heads=4, dropout=0.5,
              lr=5e-3, weight_decay=1e-4):
    model = SybilGAT(
        in_channels     = data.num_node_features,
        hidden_channels = hidden,
        out_channels    = 2,
        heads           = heads,
        dropout         = dropout,
    ).to(device)

    opt       = torch.optim.Adam(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss(weight=cw)

    best_val_f1   = 0
    best_state    = None
    patience_cnt  = 0
    PATIENCE      = 20

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        opt.zero_grad()
        out  = model(data_d.x, data_d.edge_index)
        loss = criterion(out[data_d.train_mask], data_d.y[data_d.train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        # Validate
        model.eval()
        with torch.no_grad():
            logits = model(data_d.x, data_d.edge_index)
            preds  = logits.argmax(dim=1)

            val_preds = preds[data_d.val_mask].cpu().numpy()
            val_true  = data_d.y[data_d.val_mask].cpu().numpy()
            val_f1    = f1_score(val_true, val_preds, average='macro')
            val_acc   = accuracy_score(val_true, val_preds)

        scheduler.step(1 - val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | loss {loss:.4f} | "
                  f"val_acc {val_acc:.4f} | val_f1 {val_f1:.4f} | "
                  f"patience {patience_cnt}/{PATIENCE}")

        if patience_cnt >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    # Evaluate on test set
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(data_d.x, data_d.edge_index)
        preds  = logits.argmax(dim=1).cpu().numpy()

    test_true  = data.y[data.test_mask].numpy()
    test_preds = preds[data.test_mask]
    test_acc   = accuracy_score(test_true, test_preds)
    test_f1    = f1_score(test_true, test_preds, average='macro')

    print(f"\n✓ SybilGAT  →  Acc: {test_acc:.4f}  Macro F1: {test_f1:.4f}")
    return model, test_acc, test_f1

# ── BASELINES ─────────────────────────────────────────────────────────────────
def run_baselines(data):
    X = data.x.numpy()
    y = data.y.numpy()

    X_tr = X[data.train_mask.numpy()]
    y_tr = y[data.train_mask.numpy()]
    X_te = X[data.test_mask.numpy()]
    y_te = y[data.test_mask.numpy()]

    results = {}

    # Logistic Regression
    print("Training LR...")
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    lr.fit(X_tr, y_tr)
    p = lr.predict(X_te)
    results['LR'] = (accuracy_score(y_te, p), f1_score(y_te, p, average='macro'))
    print(f"  LR     →  Acc: {results['LR'][0]:.4f}  Macro F1: {results['LR'][1]:.4f}")

    # Random Forest
    print("Training RF...")
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 n_jobs=-1, random_state=42)
    rf.fit(X_tr, y_tr)
    p = rf.predict(X_te)
    results['RF'] = (accuracy_score(y_te, p), f1_score(y_te, p, average='macro'))
    print(f"  RF     →  Acc: {results['RF'][0]:.4f}  Macro F1: {results['RF'][1]:.4f}")

    return results

# ── RUN EVERYTHING ────────────────────────────────────────────────────────────
print("\n=== Baselines ===")
baseline_results = run_baselines(data)

print("\n=== SybilGAT ===")
model, gat_acc, gat_f1 = train_gat(data, device)

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print("\n" + "="*50)
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}")
print("="*50)
for name, (acc, f1) in baseline_results.items():
    print(f"{name:<20} {acc:>10.4f} {f1:>10.4f}")
print(f"{'SybilGAT':20} {gat_acc:>10.4f} {gat_f1:>10.4f}")
print("="*50)

## 4. RF Baseline + Feature Importance

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

X = data.x.numpy()
y = data.y.numpy()

X_tr = X[data.train_mask.numpy()]
y_tr = y[data.train_mask.numpy()]
X_te = X[data.test_mask.numpy()]
y_te = y[data.test_mask.numpy()]

# 1. Check feature importances — what is RF actually using?
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                             n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)

feature_names = [
    'followers_count', 'friends_count', 'statuses_count',
    'favourites_count', 'listed_count', 'verified', 'geo_enabled',
    'default_profile', 'has_url', 'has_desc', 'name_len', 'screen_len',
    'age_days', 'ff_ratio', 'tweet_rate', 'fav_rate', 'n_tweets'
] + [f'tfidf_{i}' for i in range(200)]

importances = pd.Series(rf.feature_importances_, index=feature_names)
print("Top 15 RF features:")
print(importances.nlargest(15).to_string())

# 2. Check if any bot group is trivially separable
print("\n\nPer-group stats on key features:")
users_df['_pred'] = rf.predict(X)
for group in GROUPS:
    mask = users_df['_group'] == group
    subset = users_df[mask]
    acc = (subset['_label'] == users_df.loc[mask, '_pred']).mean()
    print(f"  {group:<30} n={mask.sum():4d}  RF_acc={acc:.3f}  "
          f"statuses_mean={subset['statuses_count'].mean():.1f}  "
          f"followers_mean={subset['followers_count'].mean():.1f}")

## 5. Attention Weight Analysis

Extracts attention weights from GAT layer 1 and categorizes by edge type.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch_geometric.nn import GATConv

# ── EXTRACT ATTENTION WEIGHTS FROM LAYER 1 ───────────────────────────────────
# We need to re-run a forward pass with return_attention_weights=True

model.eval()
model = model.to('cpu')
data_cpu = data.to('cpu')

with torch.no_grad():
    x_in = data_cpu.x   # no dropout in eval mode

    # GATConv layer 1 with attention weights returned
    _, (edge_index_out, attn_weights) = model.conv1(
        x_in,
        data_cpu.edge_index,
        return_attention_weights=True
    )

# attn_weights shape: [num_edges, num_heads]
# Average across heads
attn = attn_weights.mean(dim=1).numpy()        # [num_edges]
src  = edge_index_out[0].numpy()               # source node per edge
dst  = edge_index_out[1].numpy()               # dest node per edge
labels = data_cpu.y.numpy()                    # node labels

print(f"Edges: {len(attn):,}")
print(f"Attention weight range: [{attn.min():.4f}, {attn.max():.4f}]")

# ── CATEGORIZE EDGES ──────────────────────────────────────────────────────────
src_label = labels[src]   # 0=human, 1=bot
dst_label = labels[dst]

# Edge type: (src_type, dst_type)
hh_mask = (src_label == 0) & (dst_label == 0)  # human → human
hb_mask = (src_label == 0) & (dst_label == 1)  # human → bot
bh_mask = (src_label == 1) & (dst_label == 0)  # bot → human (camouflage)
bb_mask = (src_label == 1) & (dst_label == 1)  # bot → bot

edge_types = {
    'Human → Human': attn[hh_mask],
    'Human → Bot':   attn[hb_mask],
    'Bot → Human\n(camouflage)': attn[bh_mask],
    'Bot → Bot':     attn[bb_mask],
}

print("\nAttention weights by edge type:")
print(f"{'Edge Type':<30} {'Count':>8} {'Mean':>8} {'Std':>8} {'Median':>8}")
print("-" * 65)
for etype, vals in edge_types.items():
    label_clean = etype.replace('\n', ' ')
    print(f"{label_clean:<30} {len(vals):>8,} {vals.mean():>8.4f} "
          f"{vals.std():>8.4f} {np.median(vals):>8.4f}")

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SybilGAT — Attention Weight Analysis (cresci-2017)', fontsize=13)

colors = ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
labels_plot = list(edge_types.keys())
means  = [v.mean() for v in edge_types.values()]
stds   = [v.std()  for v in edge_types.values()]

# Left: bar chart of mean attention by edge type
ax = axes[0]
bars = ax.bar(labels_plot, means, color=colors, alpha=0.85,
               yerr=stds, capsize=5, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Attention Weight')
ax.set_title('Mean Attention by Edge Type')
ax.set_ylim(0, max(means) * 1.3)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(np.concatenate(list(edge_types.values())).mean(),
           color='gray', linestyle='--', linewidth=1, label='Global mean')
ax.legend()

# Right: violin/box plot of distributions
ax2 = axes[1]
data_plot = [v for v in edge_types.values()]
bp = ax2.boxplot(data_plot, patch_artist=True, notch=True,
                  medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax2.set_xticklabels(labels_plot, fontsize=9)
ax2.set_ylabel('Attention Weight')
ax2.set_title('Attention Distribution by Edge Type')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cresci/attention_weights.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Saved to Drive")

## 6. Statistical Significance

Mann-Whitney U test confirming Bot→Human attention suppression:
- Bot→Human < Bot→Bot: p ≈ 0
- Bot→Human < Human→Bot: p = 1.3×10⁻⁴¹

In [ ]:
from scipy import stats

bh = edge_types['Bot → Human\n(camouflage)']
bb = edge_types['Bot → Bot']
hh = edge_types['Human → Human']
hb = edge_types['Human → Bot']

stat, p = stats.mannwhitneyu(bh, bb, alternative='less')
print(f"Bot→Human < Bot→Bot:  U={stat:.0f}, p={p:.4e}")

stat2, p2 = stats.mannwhitneyu(bh, hb, alternative='less')
print(f"Bot→Human < Human→Bot: U={stat2:.0f}, p={p2:.4e}")

## 7. GCN + GraphSAGE Baselines

In [ ]:
from torch_geometric.nn import GCNConv, SAGEConv

# ── GCN ───────────────────────────────────────────────────────────────────────
class BotGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.skip  = nn.Linear(in_channels, out_channels)
        self.bn1   = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x_in = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv1(x_in, edge_index)
        x    = self.bn1(x)
        x    = F.elu(x)
        x    = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv2(x, edge_index)
        x    = x + self.skip(x_in)
        return x

# ── GraphSAGE ─────────────────────────────────────────────────────────────────
class BotSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.skip  = nn.Linear(in_channels, out_channels)
        self.bn1   = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x_in = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv1(x_in, edge_index)
        x    = self.bn1(x)
        x    = F.elu(x)
        x    = F.dropout(x, p=self.dropout, training=self.training)
        x    = self.conv2(x, edge_index)
        x    = x + self.skip(x_in)
        return x

# ── GENERIC TRAINER (reusable for all three models) ───────────────────────────
def train_model(model, data, device, epochs=300, lr=5e-3, weight_decay=1e-4):
    model     = model.to(device)
    data_d    = data.to(device)
    opt       = torch.optim.Adam(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss(weight=cw)  # cw defined earlier

    best_val_f1  = 0
    best_state   = None
    patience_cnt = 0
    PATIENCE     = 20

    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        out  = model(data_d.x, data_d.edge_index)
        loss = criterion(out[data_d.train_mask], data_d.y[data_d.train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data_d.x, data_d.edge_index)
            preds  = logits.argmax(dim=1)
            val_preds = preds[data_d.val_mask].cpu().numpy()
            val_true  = data_d.y[data_d.val_mask].cpu().numpy()
            val_f1    = f1_score(val_true, val_preds, average='macro')

        scheduler.step(1 - val_f1)

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if patience_cnt >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        preds = model(data.x.to(device), data.edge_index.to(device))
        preds = preds.argmax(dim=1).cpu().numpy()

    test_true  = data.y[data.test_mask].numpy()
    test_preds = preds[data.test_mask]
    acc = accuracy_score(test_true, test_preds)
    f1  = f1_score(test_true, test_preds, average='macro')
    return model, acc, f1

# ── RUN ALL THREE ─────────────────────────────────────────────────────────────
in_ch = data.num_node_features

print("Training GCN...")
gcn_model, gcn_acc, gcn_f1 = train_model(
    BotGCN(in_ch, 64, 2), data, device)
print(f"  GCN      →  Acc: {gcn_acc:.4f}  Macro F1: {gcn_f1:.4f}")

print("Training GraphSAGE...")
sage_model, sage_acc, sage_f1 = train_model(
    BotSAGE(in_ch, 64, 2), data, device)
print(f"  SAGE     →  Acc: {sage_acc:.4f}  Macro F1: {sage_f1:.4f}")

print("Re-training SybilGAT with shared trainer for fair comparison...")
gat_model, gat_acc, gat_f1 = train_model(
    SybilGAT(in_ch, 64, 2), data, device)
print(f"  SybilGAT →  Acc: {gat_acc:.4f}  Macro F1: {gat_f1:.4f}")

# ── FINAL TABLE ───────────────────────────────────────────────────────────────
print("\n" + "="*52)
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}")
print("="*52)
print(f"{'Logistic Regression':<20} {'0.9346':>10} {'0.9131':>10}")
print(f"{'GCN':<20} {gcn_acc:>10.4f} {gcn_f1:>10.4f}")
print(f"{'GraphSAGE':<20} {sage_acc:>10.4f} {sage_f1:>10.4f}")
print(f"{'SybilGAT':<20} {gat_acc:>10.4f} {gat_f1:>10.4f}")
print(f"{'Random Forest':<20} {'0.9879':>10} {'0.9841':>10}")
print("="*52)

## 8. GNNExplainer — Ego Network Visualization

Explains a single social_spambots_1 account — shows which neighbors and features drove the classification.

In [ ]:
import torch
from torch_geometric.explain import Explainer, GNNExplainer
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import numpy as np

# ── PICK TARGET NODE ──────────────────────────────────────────────────────────
# Find a correctly classified social_spambot_1 in test set
group_mask  = (users_df['_group'] == 'social_spambots_1').values
test_mask   = data.test_mask.numpy()
candidate_mask = group_mask & test_mask

# Get model predictions on full graph
gat_model.eval()
gat_model = gat_model.to('cpu')
with torch.no_grad():
    logits = gat_model(data.x, data.edge_index)
    preds  = logits.argmax(dim=1).numpy()

correct_mask = (preds == data.y.numpy())
final_mask   = candidate_mask & correct_mask
candidates   = np.where(final_mask)[0]
print(f"Correctly classified social_spambots_1 in test set: {len(candidates)}")

target_node = int(candidates[0])
print(f"Target node index: {target_node}")
print(f"True label: {data.y[target_node].item()} (1=bot)")
print(f"Predicted:  {preds[target_node]}")

# ── RUN GNNExplainer ──────────────────────────────────────────────────────────
explainer = Explainer(
    model=gat_model,
    algorithm=GNNExplainer(epochs=300),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='raw',
    ),
)

explanation = explainer(
    x          = data.x,
    edge_index = data.edge_index,
    index      = target_node,
)

print(f"\nExplanation generated.")
print(f"Node mask shape: {explanation.node_mask.shape}")
print(f"Edge mask shape: {explanation.edge_mask.shape}")

# ── TOP FEATURES ──────────────────────────────────────────────────────────────
feature_names = [
    'followers_count', 'friends_count', 'statuses_count',
    'favourites_count', 'listed_count', 'verified', 'geo_enabled',
    'default_profile', 'has_url', 'has_desc', 'name_len', 'screen_len',
    'age_days', 'ff_ratio', 'tweet_rate', 'fav_rate', 'n_tweets'
] + [f'tfidf_{i}' for i in range(200)]

# Node mask: importance of each feature for the target node
node_feat_importance = explanation.node_mask[target_node].numpy()
top_idx  = np.argsort(node_feat_importance)[::-1][:15]
top_names = [feature_names[i] for i in top_idx]
top_vals  = node_feat_importance[top_idx]

print("\nTop 15 features for this bot's classification:")
for name, val in zip(top_names, top_vals):
    print(f"  {name:<25} {val:.4f}")

# ── EGO NETWORK VISUALIZATION ─────────────────────────────────────────────────
# Get edges involving target node
edge_index_np = data.edge_index.numpy()
edge_mask_np  = explanation.edge_mask.numpy()

# Find edges connected to target node
connected = (edge_index_np[0] == target_node) | (edge_index_np[1] == target_node)
ego_edges  = edge_index_np[:, connected]
ego_masks  = edge_mask_np[connected]

# Unique nodes in ego network
ego_nodes  = np.unique(ego_edges)
node_labels = data.y.numpy()

# Build networkx graph
G = nx.DiGraph()
for n in ego_nodes:
    G.add_node(n, label=node_labels[n], is_target=(n == target_node))

for i in range(ego_edges.shape[1]):
    src, dst = ego_edges[0, i], ego_edges[1, i]
    G.add_edge(src, dst, weight=float(ego_masks[i]))

# Layout
pos = nx.spring_layout(G, seed=42, k=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'GNNExplainer — Social Spambot Ego Network\nNode {target_node} '
             f'(social_spambots_1)', fontsize=13)

# Left: ego network
ax = axes[0]
node_colors = []
for n in G.nodes():
    if n == target_node:
        node_colors.append('#FF0000')   # target = red
    elif node_labels[n] == 1:
        node_colors.append('#FF9800')   # other bots = orange
    else:
        node_colors.append('#2196F3')   # humans = blue

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_colors = [plt.cm.Reds(w / max_w + 0.2) for w in edge_weights]
edge_widths = [1 + 4 * (w / max_w) for w in edge_weights]

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                        node_size=300, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors,
                        width=edge_widths, ax=ax, arrows=True,
                        arrowsize=15, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G, pos, ax=ax, font_size=6)

legend = [
    mpatches.Patch(color='#FF0000', label='Target bot'),
    mpatches.Patch(color='#FF9800', label='Bot neighbor'),
    mpatches.Patch(color='#2196F3', label='Human neighbor'),
]
ax.legend(handles=legend, loc='upper left', fontsize=9)
ax.set_title('Ego Network (edge width = explainer importance)')
ax.axis('off')

# Right: top feature importances
ax2 = axes[1]
colors_feat = ['#F44336' if 'tfidf' not in n else '#9C27B0' for n in top_names]
bars = ax2.barh(top_names[::-1], top_vals[::-1], color=colors_feat[::-1],
                 edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Feature Importance (GNNExplainer)')
ax2.set_title('Top 15 Discriminative Features')
red_patch   = mpatches.Patch(color='#F44336', label='Profile feature')
purple_patch = mpatches.Patch(color='#9C27B0', label='TF-IDF feature')
ax2.legend(handles=[red_patch, purple_patch], fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cresci/gnnexplainer_ego.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Saved to Drive")s